# 3. Evaluación y trazabilidad

Ejecutar primero el notebook 01. Este notebook usa el mismo paquete/biblioteca que consumirá la UI, sobre el consolidado local. El intérprete de referencia es local; no demuestra rendimiento distribuido para 30 millones de clientes.

In [1]:
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Ejecutar desde el proyecto o notebooks/"
import json
import duckdb
from rule_manager.duckdb_backend import DuckDBBackend, json_text
from rule_manager.synthetic import ALLOWED_TABLES
from rule_manager.examples import input_definition, evaluation_definition
con = duckdb.connect(str(ROOT / "data/banking.duckdb"))
backend = DuckDBBackend(con, ALLOWED_TABLES)
package = evaluation_definition(input_definition())
report = backend.evaluate(package, "input-demo-001", "evaluation-demo-001")
assert report["rows"] == 90000
print({k:v for k,v in report.items() if k != "details"})


{'execution_id': 'evaluation-demo-001', 'status': 'completed_with_errors', 'rows': 90000, 'reused': True}


In [2]:
summary = con.execute("""
SELECT product, count(*) AS clients,
count(*) FILTER (WHERE result=true) AS accepted,
count(*) FILTER (WHERE result=false) AS denied,
count(*) FILTER (WHERE result IS NULL) AS errors
FROM evaluations GROUP BY product ORDER BY product
""").fetchall()
summary


[('basic_account', 30000, 30000, 0, 0),
 ('demo_credit', 30000, 14964, 15005, 31),
 ('savings_offer', 30000, 22494, 7506, 0)]

## Un error no es una denegación

Cada 997 clientes hay un ingreso sintético cero. La división falla, la regla dependiente y el producto de crédito quedan null con diagnóstico. La cuenta básica puede aprobarse para el mismo cliente. AND/OR no oculta una regla fallida.

In [3]:
outcomes = dict(con.execute("SELECT product,result FROM evaluations WHERE client='C000000'").fetchall())
assert outcomes["basic_account"] is True
assert outcomes["demo_credit"] is None
print(outcomes)
rule_trace = con.execute("SELECT rules FROM rule_results WHERE client='C000000'").fetchone()[0]
assert len(rule_trace) == 4
print(json_text(rule_trace))


{'basic_account': True, 'demo_credit': None, 'savings_offer': False}
[{"error_code": null, "error_message": null, "id": "adult", "result": true, "version": 1}, {"error_code": "DEPENDENCY_ERROR", "error_message": "variable.debt_ratio: DIVISION_BY_ZERO", "id": "affordability", "result": null, "version": 1}, {"error_code": null, "error_message": null, "id": "has_accounts", "result": false, "version": 1}, {"error_code": null, "error_message": null, "id": "non_negative_balance", "result": true, "version": 1}]


In [4]:
variable_trace = con.execute("SELECT variables FROM virtual_variables WHERE client='C000000'").fetchone()[0]
print(json_text(variable_trace))
assert next(v for v in variable_trace if v["id"] == "debt_ratio")["error_code"] == "DIVISION_BY_ZERO"
assert con.execute("SELECT count(*) FROM virtual_variables").fetchone()[0] == 30000
assert con.execute("SELECT count(*) FROM rule_results").fetchone()[0] == 30000


[{"data_type": "{\"nullable\": false, \"precision\": 18, \"scale\": 2, \"type\": \"decimal\"}", "dependencies": [], "error_code": null, "error_message": null, "id": "income", "value": {"boolean_value": null, "date_value": null, "decimal_value": "0.000000", "integer_value": null, "string_value": null, "timestamp_value": null}, "version": 1}, {"data_type": "{\"nullable\": false, \"precision\": 18, \"scale\": 2, \"type\": \"decimal\"}", "dependencies": [], "error_code": null, "error_message": null, "id": "debt", "value": {"boolean_value": null, "date_value": null, "decimal_value": "1049.730000", "integer_value": null, "string_value": null, "timestamp_value": null}, "version": 1}, {"data_type": "{\"nullable\": false, \"precision\": 28, \"scale\": 6, \"type\": \"decimal\"}", "dependencies": ["debt", "income"], "error_code": "DIVISION_BY_ZERO", "error_message": "Denominator is zero", "id": "debt_ratio", "value": {"boolean_value": null, "date_value": null, "decimal_value": null, "integer_valu

## Reintento idempotente

El mismo ID con las mismas entradas conserva resultados. Si cambia lógica o materialización para ese ID, la biblioteca rechaza la solicitud. Una ejecución intencional nueva requiere otro ID.

In [5]:
retry = backend.evaluate(package, "input-demo-001", "evaluation-demo-001")
assert retry["reused"] is True
assert con.execute("SELECT count(*) FROM evaluations").fetchone()[0] == 90000
print("Reintento sin duplicados.")
con.close()


Reintento sin duplicados.
